In [3]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "8cd15b981ab645ddfb817d5195df29a6bb00fcc058a15120a50d93fd6fc76a5e"
client = Together(api_key=TOGETHER_API_KEY)

def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)
                jitter = random.uniform(0, 0.5 * base_delay)
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            response = client.chat.completions.create(
                model="meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo",
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Gene Ontology (GO). Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=512,  # Increased for the larger 70B model
                response_format={"type": "json_object"}
            )

            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")

    return json.dumps({"GO_TERM": "Error"})

def extract_go_data(response_text):
    try:
        # First, try to parse as direct JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'GO_TERM' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass

        # Look for JSON in code blocks
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'GO_TERM' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Look for JSON pattern with GO_TERM
        json_pattern = re.compile(r'(\{[^{]*?"GO_TERM"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'GO_TERM' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract GO_TERM value directly
        go_term_match = re.search(r'"GO_TERM"[\s:]*"([^"]+)"', response_text)
        if go_term_match:
            return {"GO_TERM": go_term_match.group(1).strip()}

        # Check for "None" responses
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"GO_TERM": "None"}

        return {"GO_TERM": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"GO_TERM": "Parse_Error"}

def get_go_term_from_id(go_id):
    # Enhanced prompt to prefer exact/concise terms
    prompt = f"""Given the Gene Ontology ID "{go_id}", provide the corresponding GO term name.

Instructions:
- Provide the EXACT official GO term name as it appears in the Gene Ontology database
- Choose the most concise, standard form of the term
- Avoid overly specific or lengthy variations (e.g., prefer "membrane" over "integral component of membrane")
- Return the primary, commonly used GO term name
- If the GO ID is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "GO_TERM": "<go_term_name_here>"
}}

GO ID: {go_id}"""

    response_text = query_together(prompt)
    result = extract_go_data(response_text)

    log_entry = {
        "go_id": go_id,
        "raw_response": response_text,
        "extracted_result": result
    }

    with open("go_api_responses_llama31_70b.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("GO_TERM", "Error")

def calculate_match(original_go_term, new_go_term):
    if pd.isna(original_go_term) or pd.isna(new_go_term):
        return 0
    return int(str(original_go_term).strip() == str(new_go_term).strip())

def main():
    print("Starting GO ID to term mapping process with Meta-Llama-3.1-70B-Instruct-Turbo...")

    if not os.path.exists("go_api_responses_llama31_70b.jsonl"):
        open("go_api_responses_llama31_70b.jsonl", "w").close()

    print("Testing API connection...")
    test_result = get_go_term_from_id("GO:0006915")
    print(f"Test result: {test_result}")

    if test_result in ["Error", "API_Error"]:
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    input_file = "Datasets/go_terms.csv"  # Updated path to match uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    df["newgoterm"] = None
    df["match_result"] = None

    total_rows = len(df)
    for idx, (index, row) in enumerate(df.iterrows()):
        try:
            go_id = row["go_id"]
            original_go_term = row["go_term"]

            print(f"Processing {idx+1}/{total_rows}: {go_id}")
            new_go_term = get_go_term_from_id(go_id)

            df.at[index, "newgoterm"] = new_go_term
            df.at[index, "match_result"] = calculate_match(original_go_term, new_go_term)

            print(f"  Original: {original_go_term}, New: {new_go_term}, Match: {df.at[index, 'match_result']}")

            # Save progress every 10 iterations
            if idx % 10 == 0 or idx == total_rows - 1:
                excluded_columns = ['normalized_term', 'normalized_id', 'go_match', 'processing_time']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("llama31_70b_go_results_progress.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} terms.")

            # Add delay between requests (reduced for larger model - better rate limits)
            if idx < total_rows - 1:
                delay = random.uniform(1, 3)  # Shorter delay for 70B model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.at[index, "newgoterm"] = "Error"
            df.at[index, "match_result"] = 0

    # Save final results
    excluded_columns = ['normalized_term', 'normalized_id', 'go_match', 'processing_time']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    final_df.to_csv("llama31_70b_go_results_final.csv", index=False)

    print(f"\n=== FINAL RESULTS (Meta-Llama-3.1-70B-Instruct-Turbo) ===")
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)

        none_results = (df["newgoterm"] == "None").sum()
        error_results = (df["newgoterm"] == "Error").sum()
        parse_error_results = (df["newgoterm"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results

        print(f"Total terms processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {(total_matches / total_processed) * 100:.2f}%")
        print(f"\nResults breakdown:")
        print(f"  - Valid GO terms returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")

        valid_matches = df[~df["newgoterm"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
        valid_accuracy = (valid_matches / valid_results) * 100 if valid_results > 0 else 0.0
        print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")
        print(f"\nACCURACY (valid results only): {valid_accuracy:.2f}%")

        if 'Aspect' in df.columns:
            print(f"\nBreakdown by GO Aspect:")
            aspect_stats = df.groupby('Aspect').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(aspect_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: llama31_70b_go_results_final.csv")
    print(f"Progress file: llama31_70b_go_results_progress.csv")
    print(f"Log file: go_api_responses_llama31_70b.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id, go_match, processing_time")

if __name__ == "__main__":
    main()

Starting GO ID to term mapping process with Meta-Llama-3.1-70B-Instruct-Turbo...
Testing API connection...
Sending request (attempt 1/5)...
Test result: apoptotic process
API test successful, proceeding with batch processing...
Loaded 1839 rows from Datasets/go_terms.csv
Columns in the dataset: ['go_id', 'go_annotations', 'go_term', 'Definitions', 'Parents', 'Aspect', 'go_id_pmc', 'go_term_pmc', 'normalized_term', 'normalized_id', 'go_match']
Processing 1/1839: GO:0016020
Sending request (attempt 1/5)...
  Original: membrane, New: membrane, Match: 1
Progress saved. Processed 1/1839 terms.
Waiting 1.69 seconds before next request...
Processing 2/1839: GO:0005634
Sending request (attempt 1/5)...
  Original: nucleus, New: nucleus, Match: 1
Waiting 2.42 seconds before next request...
Processing 3/1839: GO:0005694
Sending request (attempt 1/5)...
  Original: chromosome, New: chromosome, Match: 1
Waiting 1.08 seconds before next request...
Processing 4/1839: GO:0005737
Sending request (attem

In [4]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "8cd15b981ab645ddfb817d5195df29a6bb00fcc058a15120a50d93fd6fc76a5e"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            
            response = client.chat.completions.create(
                model="meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo",
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Human Phenotype Ontology (HPO) terms and IDs. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=512,  # Increased for the larger 70B model
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"hpo_term": "Error"})

# Improved function to extract JSON from the LLM response for HPO term data
def extract_hpo_term_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'hpo_term' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'hpo_term' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"hpo_term"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'hpo_term' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract hpo_term value directly
        hpo_term_match = re.search(r'"hpo_term"[\s:]*"([^"]+)"', response_text)
        if hpo_term_match:
            return {"hpo_term": hpo_term_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        hpo_term_match = re.search(r'hpo_term["\':\s]+([^"\'}\s,]+)', response_text)
        if hpo_term_match:
            return {"hpo_term": hpo_term_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"hpo_term": "None"}

        # If all parsing attempts fail
        return {"hpo_term": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"hpo_term": "Parse_Error"}

# Enhanced prompt to get HPO term for a given HPO ID
def get_hpo_term_from_id(hpo_id):
    prompt = f"""Given the Human Phenotype Ontology ID "{hpo_id}", provide the corresponding HPO term name.

Instructions:
- Provide the EXACT official HPO term name as it appears in the HPO database
- Choose the most concise, standard form of the term
- Avoid overly specific or lengthy variations (e.g., prefer "Seizure" over "Generalized tonic-clonic seizure with focal onset")
- Return the primary, commonly used HPO term name
- Use standard Human Phenotype Ontology nomenclature
- If the HPO ID is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "hpo_term": "<hpo_term_name_here>"
}}

HPO ID: {hpo_id}"""

    response_text = query_together(prompt)
    result = extract_hpo_term_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "hpo_id": hpo_id,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file
    with open("hpo_id_term_api_responses_llama31_70b.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("hpo_term", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_hpo_term, new_hpo_term):
    """
    Calculate match result between original and new HPO terms
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_hpo_term) or pd.isna(new_hpo_term):
        return 0
    return int(str(original_hpo_term).strip().lower() == str(new_hpo_term).strip().lower())

# Main processing function
def main():
    print("Starting HPO ID to HPO Term mapping process with Meta-Llama-3.1-70B-Instruct-Turbo...")

    # Create log file if it doesn't exist
    if not os.path.exists("hpo_id_term_api_responses_llama31_70b.jsonl"):
        with open("hpo_id_term_api_responses_llama31_70b.jsonl", "w") as f:
            pass

    # Test the API with a sample HPO ID
    print("Testing API connection...")
    test_result = get_hpo_term_from_id("HP:0001250")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the hpo_terms CSV
    input_file = "Datasets/hpo_terms.csv"
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_hpo_term"] = None
    df["match_result"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            hpo_id = row["hpo_id"]  # Using the hpo_id column from the CSV
            original_hpo_term = row["hpo_term"]  # Original HPO term for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {hpo_id}")

            new_hpo_term = get_hpo_term_from_id(hpo_id)

            # Store the new HPO term
            df.loc[idx, "new_hpo_term"] = new_hpo_term
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_hpo_term, new_hpo_term)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_hpo_term}, New: {new_hpo_term}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_term', 'normalized_id']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("llama31_70b_hpo_id_term_results_progress.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} HPO IDs.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(1, 3)  # Shorter delay for 70B model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_hpo_term"] = "Error"
            df.loc[idx, "match_result"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_term', 'normalized_id']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("llama31_70b_hpo_id_term_results_final.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Meta-Llama-3.1-70B-Instruct-Turbo) ===")
        print(f"Total HPO IDs processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_hpo_term"] == "None").sum()
        error_results = (df["new_hpo_term"] == "Error").sum()
        parse_error_results = (df["new_hpo_term"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid HPO terms returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_hpo_term"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about PMC counts if available
        if 'hpo_id_pmc' in df.columns:
            print(f"\nHPO ID PMC Count Statistics:")
            print(f"  - Mean HPO ID PMC count: {df['hpo_id_pmc'].mean():.2f}")
            print(f"  - Median HPO ID PMC count: {df['hpo_id_pmc'].median():.2f}")
            print(f"  - Max HPO ID PMC count: {df['hpo_id_pmc'].max():.2f}")
            
            # Show correlation between PMC count and match success
            if 'match_result' in df.columns:
                # Calculate match rate by PMC count bins
                df['pmc_bin'] = pd.cut(df['hpo_id_pmc'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
                pmc_match_stats = df.groupby('pmc_bin')['match_result'].agg(['count', 'sum', 'mean']).round(3)
                pmc_match_stats.columns = ['Total', 'Matches', 'Match_Rate']
                print(f"\nMatch Rate by HPO ID PMC Count:")
                print(pmc_match_stats)

        if 'hpo_term_pmc' in df.columns:
            print(f"\nHPO Term PMC Count Statistics:")
            print(f"  - Mean HPO Term PMC count: {df['hpo_term_pmc'].mean():.2f}")
            print(f"  - Median HPO Term PMC count: {df['hpo_term_pmc'].median():.2f}")
            print(f"  - Max HPO Term PMC count: {df['hpo_term_pmc'].max():.2f}")

        # Show statistics about original hpo_match if available
        if 'hpo_match' in df.columns:
            original_match_rate = df['hpo_match'].mean() * 100
            print(f"\nOriginal HPO match rate in dataset: {original_match_rate:.2f}%")
            
            # Compare original vs new match rates
            print(f"\nComparison:")
            print(f"  - Original match rate: {original_match_rate:.2f}%")
            print(f"  - LLM match rate: {match_rate:.2f}%")

    print(f"\nProcessing completed!")
    print(f"Final results saved to: llama31_70b_hpo_id_term_results_final.csv")
    print(f"Progress file: llama31_70b_hpo_id_term_results_progress.csv")
    print(f"Log file: hpo_id_term_api_responses_llama31_70b.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id")

if __name__ == "__main__":
    main()

Starting HPO ID to HPO Term mapping process with Meta-Llama-3.1-70B-Instruct-Turbo...
Testing API connection...
Sending request (attempt 1/5)...
Test result: Seizure
API test successful, proceeding with batch processing...
Loaded 18800 rows from Datasets/hpo_terms.csv
Columns in the dataset: ['hpo_id', 'hpo_term', 'hpo_parent', 'hpo_term_pmc', 'hpo_annotations', 'hpo_id_pmc', 'normalized_term', 'normalized_id', 'hpo_match']
Processing 1/18800: HP:0001250
Sending request (attempt 1/5)...
  Original: Seizure, New: Seizure, Match: 1
Progress saved. Processed 1/18800 HPO IDs.
Waiting 1.24 seconds before next request...
Processing 2/18800: HP:0001263
Sending request (attempt 1/5)...
  Original: Global developmental delay, New: Global developmental delay, Match: 1
Waiting 2.72 seconds before next request...
Processing 3/18800: HP:5000000
Sending request (attempt 1/5)...
  Original: Anti-AK5 antibody positivity, New: None, Match: 0
Waiting 1.86 seconds before next request...
Processing 4/1880

C:\Users\sp5526s\AppData\Local\Temp\ipykernel_9992\363589689.py:271: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pmc_match_stats = df.groupby('pmc_bin')['match_result'].agg(['count', 'sum', 'mean']).round(3)


In [5]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "8cd15b981ab645ddfb817d5195df29a6bb00fcc058a15120a50d93fd6fc76a5e"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            
            response = client.chat.completions.create(
                model="meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo",
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=512,  # Increased for the larger 70B model
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"protein_name": "Error"})

# Improved function to extract JSON from the LLM response for gene symbol to protein name data
def extract_protein_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'protein_name' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"protein_name"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract protein_name value directly
        protein_match = re.search(r'"protein_name"[\s:]*"([^"]+)"', response_text)
        if protein_match:
            return {"protein_name": protein_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        protein_match = re.search(r'protein_name["\':\s]+([^"\'}\s,]+)', response_text)
        if protein_match:
            return {"protein_name": protein_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"protein_name": "None"}

        # If all parsing attempts fail
        return {"protein_name": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"protein_name": "Parse_Error"}

# Enhanced prompt to get protein name for a given HUGO Gene Symbol
def get_protein_name_from_gene_symbol(gene_symbol):
    prompt = f"""Given the HUGO Gene Symbol "{gene_symbol}", provide the corresponding protein name.

Instructions:
- Provide the EXACT official protein name as commonly used in scientific literature
- Choose the most concise, standard form of the protein name
- Avoid overly technical or lengthy variations (e.g., prefer "p53" over "Cellular tumor antigen p53")
- Return the primary, commonly used protein name
- Use standard nomenclature when available
- For well-known proteins, use the common name (e.g., "Insulin" rather than technical variants)
- If the gene symbol is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "protein_name": "<protein_name_here>"
}}

Gene Symbol: {gene_symbol}"""

    response_text = query_together(prompt)
    result = extract_protein_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "gene_symbol": gene_symbol,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file
    with open("gene_protein_api_responses_llama31_70b.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("protein_name", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_protein, new_protein):
    """
    Calculate match result between original and new protein names
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_protein) or pd.isna(new_protein):
        return 0
    return int(str(original_protein).strip().lower() == str(new_protein).strip().lower())

# Main processing function
def main():
    print("Starting Gene Symbol to Protein Name mapping process with Meta-Llama-3.1-70B-Instruct-Turbo...")

    # Create log file if it doesn't exist
    if not os.path.exists("gene_protein_api_responses_llama31_70b.jsonl"):
        with open("gene_protein_api_responses_llama31_70b.jsonl", "w") as f:
            pass

    # Test the API with a sample gene symbol
    print("Testing API connection...")
    test_result = get_protein_name_from_gene_symbol("TP53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein_name_GN CSV
    input_file = "Datasets/protein_name_GN.csv"
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_protein_name"] = None
    df["match_result"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            gene_symbol = row["GN"]  # Using the GN column as input (gene symbol)
            original_protein = row["protein_name"]  # Original protein name for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {gene_symbol}")

            new_protein_name = get_protein_name_from_gene_symbol(gene_symbol)

            # Store the new protein name
            df.loc[idx, "new_protein_name"] = new_protein_name
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_protein, new_protein_name)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_protein}, New: {new_protein_name}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("llama31_70b_gene_protein_v2_results_progress.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} gene symbols.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(1, 3)  # Shorter delay for 70B model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_protein_name"] = "Error"
            df.loc[idx, "match_result"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("llama31_70b_gene_protein_v2_results_final.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Meta-Llama-3.1-70B-Instruct-Turbo) ===")
        print(f"Total gene symbols processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_protein_name"] == "None").sum()
        error_results = (df["new_protein_name"] == "Error").sum()
        parse_error_results = (df["new_protein_name"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid protein names returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_protein_name"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

        # Show some statistics about PMC counts if available
        if 'pmc_GN' in df.columns:
            print(f"\nPMC GN Count Statistics:")
            print(f"  - Mean PMC GN count: {df['pmc_GN'].mean():.2f}")
            print(f"  - Median PMC GN count: {df['pmc_GN'].median():.2f}")
            print(f"  - Max PMC GN count: {df['pmc_GN'].max():.2f}")

        if 'PMC_protein_name' in df.columns:
            print(f"\nPMC Protein Name Count Statistics:")
            print(f"  - Mean PMC protein name count: {df['PMC_protein_name'].mean():.2f}")
            print(f"  - Median PMC protein name count: {df['PMC_protein_name'].median():.2f}")
            print(f"  - Max PMC protein name count: {df['PMC_protein_name'].max():.2f}")

    print(f"\nProcessing completed!")
    print(f"Final results saved to: llama31_70b_gene_protein_v2_results_final.csv")
    print(f"Progress file: llama31_70b_gene_protein_v2_results_progress.csv")
    print(f"Log file: gene_protein_api_responses_llama31_70b.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

if __name__ == "__main__":
    main()

Starting Gene Symbol to Protein Name mapping process with Meta-Llama-3.1-70B-Instruct-Turbo...
Testing API connection...
Sending request (attempt 1/5)...
Test result: p53
API test successful, proceeding with batch processing...
Loaded 3978 rows from Datasets/protein_name_GN.csv
Columns in the dataset: ['ID', 'AC', 'GN', 'protein_name', 'GO_F Count', 'GO_P Count', 'GO_C Count', 'go_counts', 'bin', 'normalized_protein', 'normalized_gene_name', 'Match_GN', 'pmc_GN', 'PMC_protein_name']
Processing 1/3978: EXD2
Sending request (attempt 1/5)...
  Original: Exonuclease 3'-5' domain-containing protein 2 , New: Exonuclease 3'-5' domain-containing protein 2, Match: 1
Progress saved. Processed 1/3978 gene symbols.
Waiting 2.08 seconds before next request...
Processing 2/3978: EXOSC6
Sending request (attempt 1/5)...
  Original: Exosome complex component MTR3, New: Exosome complex component RRP47, Match: 0
Waiting 1.80 seconds before next request...
Processing 3/3978: INAVA
Sending request (attempt